# L37 · DFT 暴力实现 — `X[k]=Σ x[n]e^{-2πikn/N}`，O(N²) 双循环 + numpy 对齐验证

**目标**：实现 O(N²) 朴素 DFT，理解"每个频点（frequency bin） = 信号与一组旋转因子序列的点积"

🔗 Aurora 连接：`src/aurora/audio/transforms.py` → `dft()` 是本节的标准答案

DFT 做的事情本质上就是：用 N 个不同频率的复指数（complex exponential）序列逐一"测量"信号，测量结果（点积）就是该频率在信号里的强度与相位。把 N 次测量叠成矩阵，乘一次就得到整个频谱。朴素实现是 O(N²)，FFT 是在此基础上利用对称性降到 O(N log N)。

---

### 先手算一遍 4 点 DFT，再看公式

取 `x = [1, 0, -1, 0]`——这是频率 k=1 的余弦波（4 点采样一个完整周期）。DFT 应该在 k=1 处给出最大幅度。

手算 X[k=1]：
```
X[1] = x[0]*e^0  +  x[1]*e^{-2πi/4}  +  x[2]*e^{-2πi·2/4}  +  x[3]*e^{-2πi·3/4}
     = 1*(1)     +  0*(-i)            +  (-1)*(-1)           +  0*(i)
     = 1 + 0 + 1 + 0 = 2
```

`|X[1]| = 2` 是四个频点里最大的，X[0]=X[2]=X[3]=0——DFT 精准"听出"了这个频率。

**几何解释**：`X[k]` 是信号与"转 k 圈的复数基序列"的内积。若信号本身也在转 k 圈，步调一致，N 步叠加不抵消，幅度大；若转速不同，N 步叠加正负相消，趋于 0。这就是为什么 DFT 能把混叠的多频率分量分开：**频率相同则叠加，频率不同则相消**。

In [ ]:
import numpy as np
import time

## 定位图：N=8 DFT 矩阵全貌

先看这张热力图再推公式。每一行是一个**余弦/正弦基向量**；`X[k]` 就是信号 `x` 与第 `k` 行的点积。矩阵共 N² 次乘法——这正是 O(N²) 的来源，也是 FFT 要消灭的冗余。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

N = 8
k, n = np.mgrid[0:N, 0:N]
F = np.exp(-2j * np.pi * k * n / N)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
for ax, data, title, cmap in zip(
    axes, [F.real, F.imag],
    ['Re(F)  余弦基', 'Im(F)  负正弦基'],
    ['RdBu_r', 'PuOr_r'],
):
    im = ax.imshow(data, cmap=cmap, vmin=-1, vmax=1)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xlabel('样本 n'); ax.set_ylabel('频点 k')
    ax.set_xticks(range(N)); ax.set_yticks(range(N))
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle('N=8 DFT 矩阵  F[k,n] = exp(−2πi·k·n/8)\n'
             '第 k 行与信号 x 的点积 = X[k]（频点 k 的强度与相位）',
             fontsize=10.5, fontweight='bold')
plt.tight_layout()
plt.show()

## 1. DFT 公式与矩阵形式

单个频点的公式：

```
X[k] = sum_{n=0}^{N-1} x[n] * exp(-2*pi*i*k*n/N)
```

向量化写法：令 `w_k = [exp(-2*pi*i*k*n/N) for n in 0..N-1]`，则 `X[k] = dot(w_k, x)`。

把 k=0..N-1 所有行叠起来就是 **DFT 矩阵** `F`，满足 `X = F @ x`。`F` 是酉矩阵（除以 `sqrt(N)` 后），即 `F_normalized @ F_normalized.conj().T = I`。这意味着 DFT 是一次正交变换（orthogonal transform）——不同频率的基向量彼此垂直。

In [ ]:
# 演示：构造 4×4 DFT 矩阵并验证其酉性
N = 4
k = np.arange(N).reshape(N, 1)   # 列向量
n = np.arange(N).reshape(1, N)   # 行向量
F = np.exp(-2j * np.pi * k * n / N)
F_norm = F / np.sqrt(N)

# 验证酉性：F_norm @ F_norm^H ≈ I
residual = np.abs(F_norm @ F_norm.conj().T - np.eye(N)).max()
print(f"DFT矩阵（N=4）：")
print(np.round(F, 2))
print(f"\n酉性残差（应≈0）：{residual:.2e}")

## 2. 旋转因子 W_N

定义基本旋转因子：

```
W_N = exp(-2*pi*i / N)
```

矩阵元素可以写成：

```
F[k, n] = W_N^{k*n}
```

`W_N^{k*n}` 是单位圆上**逆时针转了 k*n/N 圈**的点。随着 k 增大，每个频率基向量转得更快，对应更高频率的振荡。`W_N` 是 N 次单位根，满足 `W_N^N = 1`，这是 FFT 蝶形运算利用的对称性根源。

In [ ]:
# 演示：N=8 的旋转因子在单位圆上的位置
N = 8
W = np.exp(-2j * np.pi / N)
roots = [W**k for k in range(N)]
print(f"N={N} 的旋转因子（k=0..{N-1}）：")
for k, w in enumerate(roots):
    print(f"  W_8^{k} = {w.real:+.4f} {w.imag:+.4f}i  |模|={abs(w):.4f}")
print(f"\nW_N^N = {W**N:.6f}（应为 1+0i）")

## 3. 频率含义与共轭对称（conjugate symmetry）

频谱 `X[k]` 对应的物理频率（Hz）：

```
f_k = k * sr / N
```

其中 `sr` 是采样率（sample rate，sr）。特殊情况：
- `k=0`：**直流分量（DC component）**，等于信号均值乘以 N
- `k=1..N//2`：正频率（有效频率范围到奈奎斯特 `sr/2`）
- `k=N//2+1..N-1`：负频率（冗余）

实信号必然满足**共轭对称**：

```
X[k] = conj(X[N-k])
```

所以实信号的独立信息只在前 `N//2+1` 个频点，这是后续 RFFT 优化的基础。

In [ ]:
# 演示：直流分量 + 共轭对称
x = np.array([1.0, 2.0, 3.0, 4.0])
X = np.fft.fft(x)
N = len(x)
print(f"x = {x}")
print(f"X = {np.round(X, 4)}")
print(f"\nX[0]（直流）= {X[0].real:.1f}，sum(x) = {sum(x):.1f}")
print(f"\n共轭对称验证：X[k] == conj(X[N-k])")
for k in range(1, N):
    match = np.isclose(X[k], np.conj(X[N-k]))
    print(f"  X[{k}] vs conj(X[{N-k}]): {match}")

## 4. ✏️ 实现 `naive_dft(x)`

**推理路线**：
1. `N = len(x)`，预分配 `X = np.zeros(N, dtype=complex)`
2. 外层循环 `k = 0..N-1`；内层向量化：用 `np.arange(N)` 生成 n 序列，算旋转因子 `twiddles = exp(-2*pi*i*k*n/N)`，然后 `X[k] = np.dot(x, twiddles)`
3. 返回 shape `(N,)` 的复数数组

**参考输入输出**：

```
x = [1, 2, 3, 4]
naive_dft(x) → [10.+0.j, -2.+2.j, -2.+0.j, -2.-2.j]
np.fft.fft([1,2,3,4]) → [10.+0.j, -2.+2.j, -2.+0.j, -2.-2.j]
误差 < 1e-10
```

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


In [ ]:
def naive_dft(x: np.ndarray) -> np.ndarray:
    """朴素 O(N²) DFT：X[k] = sum_n x[n]*exp(-2*pi*i*k*n/N)"""
    x = np.asarray(x, dtype=complex)
    N = len(x)
    X = np.zeros(N, dtype=complex)
    n = np.arange(N)
    for k in range(N):
        # ✏️ TODO: 计算旋转因子 twiddles = exp(-2*pi*i*k*n/N)
        # ✏️ TODO: X[k] = dot(x, twiddles)
        pass
    return X

In [ ]:
# 检查：与 numpy FFT 对比
x = np.array([1.0, 2.0, 3.0, 4.0])
_out = naive_dft(x)
if _out is None or not np.any(_out):
    print("⬜ 请先实现 naive_dft 的 TODO 项，再运行此格")
else:
    np.testing.assert_allclose(_out, np.fft.fft(x), atol=1e-10)
    print("✅ naive_dft([1,2,3,4]) 与 np.fft.fft 误差 < 1e-10")
    for N in [8, 16, 32]:
        xr = np.random.randn(N)
        np.testing.assert_allclose(naive_dft(xr), np.fft.fft(xr), atol=1e-10)
    print(f"✅ 随机输入 N=8,16,32 全部通过")


## 5. 参数实验：O(N²) 时间复杂度验证

对 `N = 8, 64, 256` 计时 `naive_dft`：
- N 翻 8 倍 → 时间预期翻 **64 倍**（N² 规律）
- L38 实现 FFT 后，N=256 将从 ~毫秒级降到 ~微秒级（约 1000x 加速）

In [ ]:
results = {}
for N in [8, 64, 256]:
    xr = np.random.randn(N).astype(complex)
    t0 = time.perf_counter()
    for _ in range(20):
        naive_dft(xr)
    elapsed = (time.perf_counter() - t0) / 20 * 1000  # ms
    results[N] = elapsed
    print(f"N={N:4d}: {elapsed:.4f} ms")

# 验证 N²缩放（粗略）
ratio_64_8   = results[64]  / results[8]
ratio_256_64 = results[256] / results[64]
print(f"\n时间比 N=64/N=8   = {ratio_64_8:.1f}x（理论 64x）")
print(f"时间比 N=256/N=64  = {ratio_256_64:.1f}x（理论 16x）")

---
→ **完成实现后，打开 [L42 · FFT 图形化](L42_visual_fft.ipynb)，对照蝴蝶图与纯音/和弦/噪声的频谱形态，检验你的 `naive_dft` 输出是否符合直觉，再回来。**

## 本课收束

`naive_dft(x)` 将长度 N 的信号映射为 N 个复数频谱系数 `X[k]`，输出的每个分量对应信号在频率 `k*sr/N` Hz 上的幅度与相位。该实现直接对应 Aurora 中 `src/aurora/audio/transforms.py` 的 `dft()` 函数，是后续 L39 FFT（蝶形运算）和 L43 STFT（短时滑窗）的算法基础。

下一课（L38）把 DFT 的 O(N²) 双循环加速为 O(N log N) 的蝶形分治（butterfly），这就是 FFT 的核心思想。